# ResNet + Deformable Convolution v2 (DCNv2)

**Paper:** Zhu, X., Hu, H., Lin, S., & Dai, J. (2019). Deformable ConvNets v2: More Deformable, Better Results. *CVPR 2019*. https://arxiv.org/abs/1811.11168

**Original DCNv1:** Dai, J., et al. (2017). Deformable Convolutional Networks. *ICCV 2017*. https://arxiv.org/abs/1703.06211

**Implementation:** `hale_core/nn/vision/cnn/resnet_dcn.py`

> Standard convolutions sample a fixed rectangular grid. Deformable convolutions learn per-position offsets that adapt the sampling pattern to the geometry of each image, enabling better modelling of non-rigid transformations.

## Architecture Overview

```
ResNet-50 DCN:  Same as ResNet-50 except stages 2 & 3 use DCNBottleneck

Input [B,3,H,W]
  → Stem (7×7 conv, MaxPool)     → [B,  64, H/4, W/4]
  → Stage 0: Bottleneck ×3       → [B, 256, H/4, W/4]   (standard conv)
  → Stage 1: Bottleneck ×4       → [B, 512, H/8, W/8]   (standard conv)
  → Stage 2: DCNBottleneck ×6    → [B,1024, H/16,W/16]  ← deformable 3×3
  → Stage 3: DCNBottleneck ×3    → [B,2048, H/32,W/32]  ← deformable 3×3
  → GlobalAvgPool → [B, 2048]

Inside DCNBottleneck:
  x → 1×1 BN+ReLU (squeeze)
    → ┌─────────────────────────────────────────────┐
      │  DeformConv2d (replaces standard 3×3 conv)  │
      │  ┌──────────────────────────────────────┐   │
      │  │ offset_conv(x) → [B, 2k², H, W]     │   │ → Δy, Δx per grid point
      │  │ mask_conv(x)   → [B, k², H, W]      │   │ → sigmoid weights [0,1]
      │  │ grid_sample(x, base+offset)           │   │ → bilinear sampling
      │  │ einsum(sampled * mask, weight)        │   │ → output features
      │  └──────────────────────────────────────┘   │
      └─────────────────────────────────────────────┘
    → BN+ReLU
    → 1×1 BN (expand)
    → (+skip) → ReLU

Key difference DCNv1 vs DCNv2:
  DCNv1: learns offsets only (Δy, Δx) — 2k² extra channels
  DCNv2: learns offsets + modulation masks — 2k² + k² extra channels
         The mask gates each sample: low mask → ignore noisy context
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(17, 10))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117')
    ax.axis('off')

def box(ax, x, y, w, h, label, sublabel='', color='#1f6feb', fontsize=9):
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08',
                           facecolor=color, edgecolor='#58a6ff', linewidth=1.4)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2 + (0.10 if sublabel else 0), label,
            ha='center', va='center', color='white', fontsize=fontsize,
            fontweight='bold', fontfamily='monospace')
    if sublabel:
        ax.text(x + w/2, y + h/2 - 0.17, sublabel,
                ha='center', va='center', color='#8b949e', fontsize=7)

def arr(ax, x1, y1, x2, y2, c='#58a6ff'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=c, lw=1.5))

# ── LEFT: Standard 3×3 conv grid vs Deformable ──────────────────────────────
ax = axes[0]
ax.set_xlim(0, 11)
ax.set_ylim(0, 11)
ax.set_title('Standard vs Deformable Sampling Grid', color='#f0f6fc',
             fontsize=11, fontweight='bold', pad=8)

# Standard grid (fixed 3x3)
ax.text(2.5, 10.5, 'Standard Conv (fixed grid)', ha='center',
        color='#58a6ff', fontsize=9, fontweight='bold')
for i in range(3):
    for j in range(3):
        rect = FancyBboxPatch((0.5 + j*1.4, 7.5 + i*1.0), 1.1, 0.8,
                               boxstyle='round,pad=0.05',
                               facecolor='#1f6feb', edgecolor='#388bfd', lw=1.5)
        ax.add_patch(rect)
        ax.text(1.05 + j*1.4, 7.9 + i*1.0, f'({i},{j})',
                ha='center', va='center', color='white', fontsize=7)

ax.text(2.5, 7.1, 'Each sample point fixed at\nregular k×k grid offsets',
        ha='center', color='#8b949e', fontsize=8)

# Deformable grid
ax.text(7.5, 10.5, 'Deformable Conv (learned)', ha='center',
        color='#f85149', fontsize=9, fontweight='bold')

# irregular sampling points
offsets = [(-0.3, 0.2), (0.5, -0.3), (0.2, 0.4),
           (-0.4, -0.2), (0.0, 0.1), (0.3, 0.3),
           (0.1, -0.4), (-0.2, 0.3), (0.4, -0.1)]
for i in range(3):
    for j in range(3):
        idx = i*3 + j
        ox, oy = offsets[idx]
        base_x = 5.0 + j*1.4 + ox
        base_y = 7.5 + i*1.0 + oy
        # Original position (ghost)
        rect0 = FancyBboxPatch((5.0 + j*1.4, 7.5 + i*1.0), 1.0, 0.7,
                                boxstyle='round,pad=0.05',
                                facecolor='none', edgecolor='#30363d', lw=1, linestyle='--')
        ax.add_patch(rect0)
        # Shifted position
        rect1 = FancyBboxPatch((base_x, base_y), 1.0, 0.7,
                                boxstyle='round,pad=0.05',
                                facecolor='#da3633', edgecolor='#f85149', lw=1.5)
        ax.add_patch(rect1)
        ax.text(base_x + 0.5, base_y + 0.35, f'Δ({ox:.1f},{oy:.1f})',
                ha='center', va='center', color='white', fontsize=6)

ax.text(7.5, 7.1, 'Each point shifted by learned\noffset (Δx, Δy) + modulated by mask',
        ha='center', color='#8b949e', fontsize=8)

# Divider
ax.axvline(5.5, color='#30363d', lw=1, linestyle=':')

# ── RIGHT: DeformConv2d pipeline ──────────────────────────────────────────────
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.set_title('DeformConv2d Forward Pass', color='#f0f6fc',
             fontsize=11, fontweight='bold', pad=8)

box(ax, 3, 10.8, 4, 0.6, 'Input x', '[B, C, H, W]', color='#21262d')
arr(ax, 5, 10.8, 5, 10.2)

# Two parallel branches
box(ax, 0.5, 9.0, 3.5, 0.9, 'offset_conv', '3×3 conv\n→ [B, 2k², H, W]', color='#6e40c9')
box(ax, 5.5, 9.0, 3.5, 0.9, 'mask_conv', '3×3 conv\n→ [B, k², H, W]\n→ sigmoid', color='#6e40c9')

ax.annotate('', xy=(2.0, 9.9), xytext=(5, 10.8),
            arrowprops=dict(arrowstyle='->', color='#8b949e', lw=1.5,
                            connectionstyle='arc3,rad=0.2'))
ax.annotate('', xy=(7.5, 9.9), xytext=(5, 10.8),
            arrowprops=dict(arrowstyle='->', color='#8b949e', lw=1.5,
                            connectionstyle='arc3,rad=-0.2'))

# Build grid
box(ax, 2.0, 7.6, 5.5, 0.9, 'Build Sampling Grid', 'base_grid + offsets → norm [-1,1]\n[B, H_out, W_out * k², 2]', color='#1f6feb')
arr(ax, 2.0, 9.0, 4.0, 8.5)
arr(ax, 7.5, 9.0, 6.0, 8.5)

# grid_sample
box(ax, 2.0, 6.2, 5.5, 1.0, 'F.grid_sample', 'Bilinear interpolation\n→ [B, C, H_out, W_out * k²]', color='#2ea043')
arr(ax, 5, 7.6, 5, 7.2)
ax.annotate('', xy=(2.5, 6.7), xytext=(5, 10.8),
            arrowprops=dict(arrowstyle='->', color='#58a6ff', lw=1.5,
                            connectionstyle='arc3,rad=0.4'))

# modulate
box(ax, 2.0, 4.8, 5.5, 0.9, 'Apply Masks', 'sampled × masks\n→ weighted samples', color='#da3633')
arr(ax, 5, 6.2, 5, 5.7)
ax.annotate('', xy=(6.0, 5.2), xytext=(8.5, 9.0),
            arrowprops=dict(arrowstyle='->', color='#8b949e', lw=1.5,
                            connectionstyle='arc3,rad=-0.25'))

# einsum
box(ax, 2.0, 3.4, 5.5, 0.9, 'Einsum (apply weights)', 'bhwpc,ocp→bhwo\n→ [B, out_channels, H_out, W_out]', color='#388bfd')
arr(ax, 5, 4.8, 5, 4.3)

# output
box(ax, 2.5, 2.3, 4.5, 0.7, 'Output', '[B, out_channels, H\', W\']', color='#21262d')
arr(ax, 5, 3.4, 5, 3.0)

ax.text(5, 1.8, 'offset_conv weights zero-initialized\n→ model starts as regular conv, learns deformability', 
        ha='center', color='#8b949e', fontsize=8)

plt.suptitle('Deformable Convolution v2 Architecture', fontsize=14,
             color='#f0f6fc', fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()


## Key Innovations

### 1. Problem: Fixed Receptive Fields Hurt Geometric Modelling
Standard convolutions always sample a regular k×k grid — perfect for upright rectangles, but inefficient for objects with arbitrary shapes, poses, or scales. The network compensates by learning many filters, but this is wasteful.

### 2. DCNv1: Learnable Offsets
For each output location, learn 2×k² offset values (Δy_p, Δx_p for each of the k² kernel positions). These shift where the input is sampled. The sampling is differentiable via bilinear interpolation, so offsets are trained end-to-end.

### 3. DCNv2: Adding Modulation Masks
DCNv2 adds k² modulation scalars m_p ∈ (0,1) (via sigmoid). Each sampling point's contribution is gated: the network can effectively "ignore" context outside the object of interest (e.g., background clutter), not just move the sample point.

The output formula:
```
y(p) = Σ_k w_k · x(p + p_k + Δp_k) · m_k
```
where p_k is the regular grid offset and Δp_k is the learned additional offset.

### 4. Implementation via `F.grid_sample`
Rather than requiring a compiled CUDA extension (like mmcv's `deform_conv2d`), this implementation:
1. Computes absolute sampling coordinates in pixel space
2. Normalises to [−1, 1] for `F.grid_sample`
3. Runs bilinear sampling via `F.grid_sample`
4. Applies modulation masks and weight einsum

**Zero-init of offset_conv**: This makes the model a standard ResNet at the start of training. It gradually learns deformability as needed — prevents instability from random initial offsets.

### 5. Where to Apply DCN (Stage Choice)
DCN is applied in **stages 2 and 3** (strides 16× and 32×). At these stages:
- Feature maps are small enough that the overhead is manageable
- Objects are semantically meaningful (DCN helps most at high-level features)
- Early stages (0, 1) keep standard convolutions — low-level textures benefit from regular grids

## Component Walkthrough

### `DeformConv2d.__init__` (lines 101–151 in `resnet_dcn.py`)
```python
# Offset predictor: 2k² channels (Δy and Δx per grid point)
# Zero-initialized → starts as identity (regular conv)
self.offset_conv = nn.Conv2d(in_channels, 2 * self.k2, kernel_size, stride, padding)
nn.init.zeros_(self.offset_conv.weight)
nn.init.zeros_(self.offset_conv.bias)

# Modulation predictor: k² channels, applied through sigmoid
self.mask_conv = nn.Conv2d(in_channels, self.k2, kernel_size, stride, padding)
nn.init.zeros_(self.mask_conv.weight)   # starts at sigmoid(0) = 0.5 (equal weights)

# Main conv weights (applied to sampled features)
self.weight = nn.Parameter(torch.empty(out_channels, in_channels, k, k))
nn.init.kaiming_normal_(self.weight, mode='fan_out', nonlinearity='relu')
```

### `DeformConv2d.forward` — core sampling logic (lines 153–232)
```python
# 1. Predict offsets + masks
offsets = self.offset_conv(x)              # [B, 2k², H_out, W_out]
masks   = torch.sigmoid(self.mask_conv(x)) # [B, k², H_out, W_out]

# 2. Build base grid (pixel coordinates of each output location's kernel)
base_y = out_y[H_out, 1, 1, 1] + ky_range[1, 1, k, 1]  # [H_out, W_out, k²]
base_x = out_x[1, W_out, 1, 1] + kx_range[1, 1, 1, k]  # [H_out, W_out, k²]

# 3. Add learned offsets (split into dy, dx)
sample_y = base_y + offsets[:, :k², ...]   # [B, H_out, W_out, k²]
sample_x = base_x + offsets[:, k²:, ...]   # [B, H_out, W_out, k²]

# 4. Normalise to [-1,1] for F.grid_sample
norm_y = 2.0 * sample_y / (H - 1) - 1.0
grid = torch.stack([norm_x, norm_y], dim=-1)   # [B, H_out, W_out*k², 2]

# 5. Bilinear sampling
sampled = F.grid_sample(x, grid, mode='bilinear', align_corners=True)
# sampled: [B, C, H_out, W_out, k²] after reshape

# 6. Apply masks + conv weights
sampled = sampled * masks.unsqueeze(-1)    # gate each sample point
out = torch.einsum('bhwpc,ocp->bhwo', sampled, self.weight.view(O, C, k²))
```

### `DCNBottleneck` (lines 237–287)
```python
# Only the 3×3 conv is replaced — 1×1 convs remain standard
self.conv1 = nn.Conv2d(in_planes, planes, 1, bias=False)        # 1×1 squeeze
self.conv2 = DeformConv2d(planes, planes, kernel_size=3, ...)   # ← DCN here
self.conv3 = nn.Conv2d(planes, out_channels, 1, bias=False)     # 1×1 expand
# 1×1 convs don't benefit from deformability (operate on single pixels)
```

In [ ]:
import sys
sys.path.insert(0, '../../src')

import torch
from hale_core.nn.vision import build_vision_backbone

x = torch.randn(1, 3, 224, 224)

for name in ['resnet50', 'resnet50_dcn', 'resnet101_dcn']:
    bb = build_vision_backbone(name, img_size=224)
    bb.eval()
    with torch.no_grad():
        out = bb(x)
    params = bb.num_parameters()
    print(f'{name:15s}  pooled={tuple(out.pooled_output.shape)}')
    print(f'  feature_maps: {[tuple(f.shape) for f in out.feature_maps]}')
    print(f'  parameters: {params:,}  (vs resnet50: 25M — DCN adds ~2M offset/mask params)')
    print()


## References

1. **[Primary DCNv2]** Zhu, X., Hu, H., Lin, S., & Dai, J. (2019). **Deformable ConvNets v2: More Deformable, Better Results**. *CVPR 2019*. https://arxiv.org/abs/1811.11168

2. **[Original DCNv1]** Dai, J., Qi, H., Xiong, Y., Li, Y., Zhang, G., Hu, H., & Wei, Y. (2017). **Deformable Convolutional Networks**. *ICCV 2017*. https://arxiv.org/abs/1703.06211

3. Wang, W., et al. (2022). **InternImage**. *CVPR 2023*. — Extends to DCNv3 (separable formulation), see `internimage.ipynb`.

4. Bertasius, G., Wang, H., & Torresani, L. (2021). **Is Space-Time Attention All You Need for Video Understanding? (TimeSformer)**. — Shows deformable attention analogues in video transformers.

5. Zhu, X., Su, W., Lu, L., Li, B., Wang, X., & Dai, J. (2021). **Deformable DETR**. *ICLR 2021*. — Applies deformable attention to transformer-based object detection, O(1) sampling per head.